In [1]:
import pandas as pd
import sys
import os
import numpy as np

sys.path.append(os.path.abspath("../3_Helpers"))
from handle_timeseries_dataframe import handle_timeseries_dataframe

sys.path.append(os.path.abspath("../.."))
from optimizer import variables

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
def load_price(output_path, add_lookback_time):
    df = pd.read_csv("1_Raw_data/Electricity_price.csv")
  
    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="START_TIME",
        raw_date_format="%d/%m/%Y %H:%M",
        expected_first_timestamp=pd.to_datetime("01/10/2021 00:00",format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime("31/03/2026 23:55",format="%d/%m/%Y %H:%M"),
        df_keep_columns=[variables.province],
        keep_cols_info={
            variables.province: {
                "method" : "mean",
                "rename" : "Price"
            }
        },
        filter_first_timestamp = pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M") - pd.DateOffset(days=14) if add_lookback_time else pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
        filter_last_timestamp=pd.to_datetime(variables.operation_end_date,format="%d/%m/%Y %H:%M"),
        output_granularity=f"{variables.operation_granularity_in_minutes}min"
    )
   
    df.to_csv(output_path,index=False)
    display(df[-10:])


load_price(output_path="2_Processed_data/price_extra_14_days.csv", add_lookback_time=True)
load_price(output_path="2_Processed_data/price.csv", add_lookback_time=False)

Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max
0,0,473183,digits/digits/digits digits:digits,1,31,1,12,2021,2026,0,23,0,55


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 473184 entries, 0 to 473183
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   START_TIME  473184 non-null  datetime64[ns]
 1   NSW         473184 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 7.2 MB


None

Detected source granularities (count by interval):
- 0 days 00:05:00: 473183
Dominant source granularity: 0 days 00:05:00
Missing dates before resampling count: 0
Unique granularities detected:
- 0 days 00:05:00
Granularity report:


...

None

Detected source granularities (count by interval):
- 0 days 00:05:00: 473183
Dominant source granularity: 0 days 00:05:00
Missing dates before resampling count: 0
Unique granularities detected:
- 0 days 00:05:00
Granularity report:


,granularity,from,to
0,0 days 00:05:00,2021-10-01,2026-03-31 23:55:00


Downsampling at least part of data to 60min
Missing dates after resampling count: 0
Chronological: True
Actual length: 2160 Expected length: 271752


,Date,Price
2150,2026-03-31 14:00:00,40.806667
2151,2026-03-31 15:00:00,54.313333
2152,2026-03-31 16:00:00,71.405833
2153,2026-03-31 17:00:00,77.255833
2154,2026-03-31 18:00:00,79.960000
2155,2026-03-31 19:00:00,72.641667
2156,2026-03-31 20:00:00,64.199167
2157,2026-03-31 21:00:00,63.990833
2158,2026-03-31 22:00:00,65.192500
2159,2026-03-31 23:00:00,65.026667


In [3]:
def load_generation(output_path):
    df = pd.read_csv('1_Raw_data/Generation.csv')

    df['Date'] = pd.date_range(start = pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),periods=len(df),freq='h') 
    
    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="Date",
        raw_date_format="%Y/%m/%d %H:%M",
        expected_first_timestamp=pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime("23/12/2059 14:00",format="%d/%m/%Y %H:%M"),
        df_keep_columns=[variables.site],
        keep_cols_info={
           variables.site : {
                "method" : "mean",
                "rename" : "Generation"
            }
        },
        filter_first_timestamp=pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
        filter_last_timestamp=pd.to_datetime(variables.operation_end_date,format="%d/%m/%Y %H:%M"),
        output_granularity=f"{variables.operation_granularity_in_minutes}min"
    )

    df.to_csv(output_path,index=False)
    display(df[:10])
    
load_generation(output_path='2_Processed_data/generation.csv')

Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max,digits_6_min,digits_6_max
0,0,306599,digits-digits-digits digits:digits:digits,2026,2060,1,12,1,31,0,23,0,0,0,0


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306600 entries, 0 to 306599
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       306600 non-null  datetime64[ns]
 1   Orange 2B  306600 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 4.7 MB


None

Detected source granularities (count by interval):
- 0 days 01:00:00: 306599
Dominant source granularity: 0 days 01:00:00
Missing dates before resampling count: 0
Unique granularities detected:
- 0 days 01:00:00
Already at desired granularity. No resampling applied.
Missing dates after resampling count: 0
Chronological: True
Actual length: 271752 Expected length: 271752


,Date,Generation
0,2026-01-01 00:00:00,0.000000
1,2026-01-01 01:00:00,0.000000
2,2026-01-01 02:00:00,0.000000
3,2026-01-01 03:00:00,0.000000
4,2026-01-01 04:00:00,0.000000
5,2026-01-01 05:00:00,0.223104
6,2026-01-01 06:00:00,1.813763
7,2026-01-01 07:00:00,4.427188
8,2026-01-01 08:00:00,5.488757
9,2026-01-01 09:00:00,5.944343


In [12]:
def load_lgc(output_path):
  
    df = pd.read_csv("1_Raw_data/LGC.csv")

    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="Date",
        raw_date_format="%d/%m/%Y",
        expected_first_timestamp=pd.to_datetime("01/01/2021",format="%d/%m/%Y"),
        expected_last_timestamp=pd.to_datetime("31/12/2065",format="%d/%m/%Y"),
        df_keep_columns=["LGC price"],
        keep_cols_info={
            "LGC price": {
                "method" : "ffill",
                "rename" : "LGC"
            }
        },
        filter_first_timestamp=pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
        filter_last_timestamp=pd.to_datetime(variables.operation_end_date,format="%d/%m/%Y %H:%M"),
        output_granularity=f"{variables.operation_granularity_in_minutes}min"
    )
    
    df.to_csv(output_path,index=False)
    display(df[:10])

load_lgc(output_path='2_Processed_data/lgc.csv')


Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max
0,0,13149,digits/digits/digits,1,31,1,12,2021,2057


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13150 entries, 0 to 13149
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       13150 non-null  datetime64[ns]
 1   LGC price  13150 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 205.6 KB


None

Detected source granularities (count by interval):
- 1 days 00:00:00: 13149
Dominant source granularity: 1 days 00:00:00
Missing dates before resampling count: 3286
Missing dates before resampling first 5: [Timestamp('2057-01-02 00:00:00'), Timestamp('2057-01-03 00:00:00'), Timestamp('2057-01-04 00:00:00'), Timestamp('2057-01-05 00:00:00'), Timestamp('2057-01-06 00:00:00')]
Missing dates before resampling last 5: [Timestamp('2065-12-27 00:00:00'), Timestamp('2065-12-28 00:00:00'), Timestamp('2065-12-29 00:00:00'), Timestamp('2065-12-30 00:00:00'), Timestamp('2065-12-31 00:00:00')]
Unique granularities detected:
- 1 days 00:00:00
Granularity report:


,granularity,from,to
0,1 days,2021-01-01,2057-01-01


Upsampling at least part of data to 60min
Missing dates after resampling count: 0
Chronological: True
Actual length: 271752 Expected length: 271752


,Date,LGC
0,2026-01-01 00:00:00,6.48
1,2026-01-01 01:00:00,6.48
2,2026-01-01 02:00:00,6.48
3,2026-01-01 03:00:00,6.48
4,2026-01-01 04:00:00,6.48
5,2026-01-01 05:00:00,6.48
6,2026-01-01 06:00:00,6.48
7,2026-01-01 07:00:00,6.48
8,2026-01-01 08:00:00,6.48
9,2026-01-01 09:00:00,6.48


In [5]:
def load_tariff(output_path):
  
    df = pd.DataFrame(pd.date_range(start = pd.to_datetime(variables.operation_start_date),end = pd.to_datetime(variables.operation_end_date),freq=f"{variables.operation_granularity_in_minutes}min"),columns=['Date'])
    mapping_df_1 = pd.read_csv('1_Raw_data/Network_time_of_use_tariff_mapping.csv').melt(id_vars=["Tariff code", "Day"],var_name="Hour",value_name="ToU_label")
    mapping_df_2 = pd.read_csv('1_Raw_data/Network_tariff_values.csv')
   
    df["ToU_label"] = (
            df["Date"]
            .apply(
                lambda d: mapping_df_1.loc[
                    (mapping_df_1["Tariff code"] == variables.tariff_code) & 
                    (mapping_df_1["Hour"].astype(int) == d.hour + 1) & 
                    (mapping_df_1["Day"].str.lower() == ("weekend" if d.weekday() >= 5 else "weekday")), 
                    "ToU_label"
                ].iloc[0]
            )
        )
    
    df["Import_charge"] = (
        df
        .apply(
            lambda r: mapping_df_2.loc[
                (mapping_df_2["Tariff code"] == variables.tariff_code) & (mapping_df_2["ToU"] == r["ToU_label"]), "Import Charge"
            ].iloc[0] * 100 / 1000, axis=1
        )
    )
  
    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="Date",
        raw_date_format="%d/%m/%Y",
        expected_first_timestamp=pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime(variables.operation_end_date,format="%d/%m/%Y %H:%M"),
        df_keep_columns=["Import_charge","ToU_label"],
        keep_cols_info={
            "ToU_label": {
                "method" : "",
                "rename" : "ToU_label"
            },
            "Import_charge": {
                "method" : "",
                "rename" : "Import_charge"
            }
        },
        filter_first_timestamp=pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
        filter_last_timestamp=pd.to_datetime(variables.operation_end_date,format="%d/%m/%Y %H:%M"),
        output_granularity=f"{variables.operation_granularity_in_minutes}min"
    )
    

    df.to_csv(output_path,index=False)
    display(df[:10])
  

load_tariff(output_path='2_Processed_data/tariff.csv')

/tmp/ipykernel_18962/79616521.py:3: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.DataFrame(pd.date_range(start = pd.to_datetime(variables.operation_start_date),end = pd.to_datetime(variables.operation_end_date),freq=f"{variables.operation_granularity_in_minutes}min"),columns=['Date'])


Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max,digits_6_min,digits_6_max
0,0,271751,digits-digits-digits digits:digits:digits,2026,2056,1,12,1,31,0,23,0,0,0,0


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271752 entries, 0 to 271751
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Date           271752 non-null  datetime64[ns]
 1   Import_charge  271752 non-null  float64       
 2   ToU_label      271752 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 6.2+ MB


None

Detected source granularities (count by interval):
- 0 days 01:00:00: 271751
Dominant source granularity: 0 days 01:00:00
Missing dates before resampling count: 0
Unique granularities detected:
- 0 days 01:00:00
Already at desired granularity. No resampling applied.
Missing dates after resampling count: 0
Chronological: True
Actual length: 271752 Expected length: 271752


,Date,Import_charge,ToU_label
0,2026-01-01 00:00:00,0.0,Off-peak
1,2026-01-01 01:00:00,0.0,Off-peak
2,2026-01-01 02:00:00,0.0,Off-peak
3,2026-01-01 03:00:00,0.0,Off-peak
4,2026-01-01 04:00:00,0.0,Off-peak
5,2026-01-01 05:00:00,0.0,Off-peak
6,2026-01-01 06:00:00,0.0,Off-peak
7,2026-01-01 07:00:00,0.0,Shoulder
8,2026-01-01 08:00:00,0.0,Shoulder
9,2026-01-01 09:00:00,0.0,Shoulder


In [13]:
def load_physical_constraints(output_path):

    df = pd.DataFrame(
        pd.date_range(
            start=pd.to_datetime(variables.operation_start_date,format="%d/%m/%Y %H:%M"),
            end=(pd.to_datetime(variables.operation_end_date,format="%d/%m/%Y %H:%M") + pd.DateOffset(days=1)),
            freq="YS"
        ),
        columns=["Date"]
    )

    mapping_df_1 = (
        pd.read_csv("1_Raw_data/Physical_constraints_mapping.csv")
        .melt(
            id_vars=["Project", "Field"],
            var_name="Year",
            value_name="Value"
        )
        .assign(Year=lambda df: pd.to_numeric(df["Year"]))
        .loc[lambda df: df["Project"] == variables.bess_technology]
        .pivot(index="Year", columns="Field", values="Value")
        .sort_index()
        .dropna(how="all")
        .rename(
            index=lambda y: pd.to_datetime(
                variables.operation_start_date,
                format="%d/%m/%Y %H:%M"
            ) + pd.DateOffset(years=int(y))
        )
        .rename_axis(index="Date", columns=None)
        .reset_index()
        .iloc[:len(df)]
    )

    # Extend mapping rows by carrying forward the final known year when horizon is longer.
    mapping_df_1 = mapping_df_1.reindex(range(len(df))).ffill()

    df["bess_deg"] = mapping_df_1["BESS storage degradation"].values

    df["hte_from_pv"] = (
        mapping_df_1["DC block RTE"].values
        * mapping_df_1["DC converter"].values
        * mapping_df_1["Aux charge"].values
        / mapping_df_1["AC inverter"].values
    )

    df["hte_from_grid"] = (
        df["hte_from_pv"]
        * (mapping_df_1["AC inverter"].values ** 2)
    )

    df["hte2grid"] = (
        mapping_df_1["DC converter"].values
        * mapping_df_1["Aux discharge"].values
        * mapping_df_1["AC inverter"].values
    )

    df["pv_bess_grid"] = (
        df["hte_from_pv"]
        * df["hte2grid"]
        * mapping_df_1["AC inverter"].values
    )

    df["grid_bess_grid"] = (
        df["hte_from_grid"]
        * df["hte2grid"]
    )

    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="Date",
        raw_date_format="%d/%m/%Y %H:%M",
        expected_first_timestamp=pd.to_datetime(variables.operation_start_date, format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime(variables.operation_end_date, format="%d/%m/%Y %H:%M"),
        df_keep_columns=[
            "bess_deg",
            "hte_from_pv",
            "hte_from_grid",
            "hte2grid",
            "pv_bess_grid",
            "grid_bess_grid"
        ],
        keep_cols_info={
            "bess_deg": {
                "method": "interpolate",
                "rename": "bess_deg"
            },
            "hte_from_pv": {
                "method": "interpolate",
                "rename": "hte_from_pv"
            },
            "hte_from_grid": {
                "method": "interpolate",
                "rename": "hte_from_grid"
            },
            "hte2grid": {
                "method": "interpolate",
                "rename": "hte2grid"
            },
            "pv_bess_grid": {
                "method": "interpolate",
                "rename": "pv_bess_grid"
            },
            "grid_bess_grid": {
                "method": "interpolate",
                "rename": "grid_bess_grid"
            },
        },
        filter_first_timestamp=pd.to_datetime(variables.operation_start_date, format="%d/%m/%Y %H:%M"),
        filter_last_timestamp=pd.to_datetime(variables.operation_end_date, format="%d/%m/%Y %H:%M"),
        output_granularity=f'{variables.operation_granularity_in_minutes}min'
    )

    df.to_csv(output_path,index=False)
    display(df.head(10))

load_physical_constraints(output_path="2_Processed_data/physical_constraint.csv")

Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max,digits_6_min,digits_6_max
0,0,31,digits-digits-digits digits:digits:digits,2026,2057,1,1,1,1,0,0,0,0,0,0


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            32 non-null     datetime64[ns]
 1   bess_deg        32 non-null     float64       
 2   hte_from_pv     32 non-null     float64       
 3   hte_from_grid   32 non-null     float64       
 4   hte2grid        32 non-null     float64       
 5   pv_bess_grid    32 non-null     float64       
 6   grid_bess_grid  32 non-null     float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 1.9 KB


None

Detected source granularities (count by interval):
- 365 days 00:00:00: 23
- 366 days 00:00:00: 8
Dominant source granularity: 365 days 00:00:00
Missing dates before resampling count: 29
Missing dates before resampling first 5: [Timestamp('2028-12-31 00:00:00'), Timestamp('2029-12-31 00:00:00'), Timestamp('2030-12-31 00:00:00'), Timestamp('2031-12-31 00:00:00'), Timestamp('2032-12-30 00:00:00')]
Missing dates before resampling last 5: [Timestamp('2052-12-25 00:00:00'), Timestamp('2053-12-25 00:00:00'), Timestamp('2054-12-25 00:00:00'), Timestamp('2055-12-25 00:00:00'), Timestamp('2056-12-24 00:00:00')]
Unique granularities detected:
- 365 days 00:00:00
- 366 days 00:00:00
Granularity report:


,granularity,from,to
0,365 days,2026-01-01,2028-01-01
1,366 days,2028-01-01,2029-01-01
2,365 days,2029-01-01,2032-01-01
3,366 days,2032-01-01,2033-01-01
4,365 days,2033-01-01,2036-01-01
5,366 days,2036-01-01,2037-01-01
6,365 days,2037-01-01,2040-01-01
7,366 days,2040-01-01,2041-01-01
8,365 days,2041-01-01,2044-01-01
9,366 days,2044-01-01,2045-01-01


Upsampling at least part of data to 60min
Missing dates after resampling count: 0
Chronological: True
Actual length: 271752 Expected length: 271752


,Date,bess_deg,hte_from_pv,hte_from_grid,hte2grid,pv_bess_grid,grid_bess_grid
0,2026-01-01 00:00:00,1.000000,0.926727,0.880467,0.942151,0.851046,0.829534
1,2026-01-01 01:00:00,0.999995,0.926726,0.880467,0.942151,0.851046,0.829533
2,2026-01-01 02:00:00,0.999991,0.926726,0.880467,0.942151,0.851046,0.829533
3,2026-01-01 03:00:00,0.999986,0.926726,0.880466,0.942151,0.851045,0.829533
4,2026-01-01 04:00:00,0.999982,0.926725,0.880466,0.942151,0.851045,0.829532
5,2026-01-01 05:00:00,0.999977,0.926725,0.880466,0.942151,0.851045,0.829532
6,2026-01-01 06:00:00,0.999972,0.926725,0.880465,0.942151,0.851044,0.829532
7,2026-01-01 07:00:00,0.999968,0.926724,0.880465,0.942151,0.851044,0.829531
8,2026-01-01 08:00:00,0.999963,0.926724,0.880465,0.942151,0.851044,0.829531
9,2026-01-01 09:00:00,0.999959,0.926724,0.880464,0.942151,0.851044,0.829531


In [7]:
def load_historical_price(output_path):
    df = pd.read_csv("1_Raw_data/Electricity_price_historical.csv")
  
    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="DATETIME",
        raw_date_format="%d/%m/%Y %H:%M",
        expected_first_timestamp=pd.to_datetime("01/01/2004 0:00",format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime("31/12/2024 23:55",format="%d/%m/%Y %H:%M"),
        df_keep_columns=["RRP_NSW1"],
        keep_cols_info={
            "RRP_NSW1": {
                "method" : "mean",
                "rename" : "Price"
            }
        },
        filter_first_timestamp = pd.to_datetime("01/01/2004 00:00",format="%d/%m/%Y %H:%M"),
        filter_last_timestamp = pd.to_datetime("31/12/2024 23:55",format="%d/%m/%Y %H:%M"),
        output_granularity = f"{60}min"
    )
   
    df.to_csv(output_path,index=False)
    display(df[-10:])


load_historical_price(output_path="2_Processed_data/historical_price.csv")

Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max
0,0,653327,digits/digits/digits digits:digits,1,31,1,12,2004,2024,0,23,0,55


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 653328 entries, 0 to 653327
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   DATETIME  653328 non-null  datetime64[ns]
 1   RRP_NSW1  653328 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 10.0 MB


None

Detected source granularities (count by interval):
- 0 days 00:05:00: 342143
- 0 days 00:30:00: 311184
Dominant source granularity: 0 days 00:05:00
Missing dates before resampling count: 1555920
Missing dates before resampling first 5: [Timestamp('2004-01-01 00:05:00'), Timestamp('2004-01-01 00:10:00'), Timestamp('2004-01-01 00:15:00'), Timestamp('2004-01-01 00:20:00'), Timestamp('2004-01-01 00:25:00')]
Missing dates before resampling last 5: [Timestamp('2021-09-30 23:35:00'), Timestamp('2021-09-30 23:40:00'), Timestamp('2021-09-30 23:45:00'), Timestamp('2021-09-30 23:50:00'), Timestamp('2021-09-30 23:55:00')]
Unique granularities detected:
- 0 days 00:05:00
- 0 days 00:30:00
Granularity report:


,granularity,from,to
0,0 days 00:30:00,2004-01-01,2021-10-01 00:00:00
1,0 days 00:05:00,2021-10-01,2024-12-31 23:55:00


Downsampling at least part of data to 60min
Missing dates after resampling count: 0
Chronological: True
Actual length: 184104 Expected length: 184104


,Date,Price
184094,2024-12-31 14:00:00,79.085833
184095,2024-12-31 15:00:00,125.118333
184096,2024-12-31 16:00:00,92.017500
184097,2024-12-31 17:00:00,56.513333
184098,2024-12-31 18:00:00,85.591667
184099,2024-12-31 19:00:00,96.520000
184100,2024-12-31 20:00:00,79.415833
184101,2024-12-31 21:00:00,97.800833
184102,2024-12-31 22:00:00,108.400000
184103,2024-12-31 23:00:00,128.565833


In [8]:
def load_historical_generation(output_path):
    df = pd.read_csv('1_Raw_data/Historical_generation.csv')
    
    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="Date",
        raw_date_format="%d/%m/%Y %H:%M",
        expected_first_timestamp=pd.to_datetime("01/01/2007 00:00",format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime("31/12/2019 23:00",format="%d/%m/%Y %H:%M"),
        df_keep_columns=["kW"],
        keep_cols_info={
           "kW" : {
                "method" : "mean",
                "rename" : "Generation"
            }
        },
        filter_first_timestamp=pd.to_datetime("01/01/2007 00:00",format="%d/%m/%Y %H:%M"),
        filter_last_timestamp=pd.to_datetime("31/12/2019 23:00",format="%d/%m/%Y %H:%M"),
        output_granularity=f"{variables.operation_granularity_in_minutes}min"
    )

    df.to_csv(output_path,index=False)
    display(df[:10])
    
load_historical_generation(output_path='2_Processed_data/historical_generation.csv')

Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max
0,0,113951,digits/digits/digits digits:digits,1,31,1,12,2007,2019,0,23,0,0


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113952 entries, 0 to 113951
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   Date    113952 non-null  datetime64[ns]
 1   kW      113952 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.7 MB


None

Detected source granularities (count by interval):
- 0 days 01:00:00: 113951
Dominant source granularity: 0 days 01:00:00
Missing dates before resampling count: 0
Unique granularities detected:
- 0 days 01:00:00
Already at desired granularity. No resampling applied.
Missing dates after resampling count: 0
Chronological: True
Actual length: 113952 Expected length: 113952


,Date,Generation
0,2007-01-01 00:00:00,0.0
1,2007-01-01 01:00:00,0.0
2,2007-01-01 02:00:00,0.0
3,2007-01-01 03:00:00,0.0
4,2007-01-01 04:00:00,0.0
5,2007-01-01 05:00:00,0.0
6,2007-01-01 06:00:00,0.0
7,2007-01-01 07:00:00,0.0
8,2007-01-01 08:00:00,0.0
9,2007-01-01 09:00:00,0.0


In [9]:
def load_future_price(output_path):
    df = pd.read_csv("1_Raw_data/Future_electricity_price_forecasts/R&M forecast prices 30m.csv")
  
    df = handle_timeseries_dataframe(
        df=df,
        date_column_name="Start Time",
        raw_date_format="%d/%m/%Y %H:%M",
        expected_first_timestamp=pd.to_datetime("01/01/2025 0:00",format="%d/%m/%Y %H:%M"),
        expected_last_timestamp=pd.to_datetime("30/06/2060 23:30",format="%d/%m/%Y %H:%M"),
        df_keep_columns=["NSW - Aurora"],
        keep_cols_info={
            "NSW - Aurora": {
                "method" : "mean",
                "rename" : "Price"
            }
        },
        filter_first_timestamp = pd.to_datetime("01/01/2026 00:00",format="%d/%m/%Y %H:%M"),
        filter_last_timestamp = pd.to_datetime("31/12/2056 23:00",format="%d/%m/%Y %H:%M"),
        output_granularity = f"{60}min"
    )
   
    df.to_csv(output_path,index=False)
    display(df[-10:])


load_future_price(output_path="2_Processed_data/future_price.csv")

Date report:


,from_row,to_row,structure,digits_1_min,digits_1_max,digits_2_min,digits_2_max,digits_3_min,digits_3_max,digits_4_min,digits_4_max,digits_5_min,digits_5_max
0,0,622319,digits/digits/digits digits:digits,1,31,1,12,2025,2060,0,23,0,30


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622320 entries, 0 to 622319
Data columns (total 2 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Start Time    622320 non-null  datetime64[ns]
 1   NSW - Aurora  622300 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 9.5 MB


None

Detected source granularities (count by interval):
- 0 days 00:30:00: 622319
Dominant source granularity: 0 days 00:30:00
Missing dates before resampling count: 0
Unique granularities detected:
- 0 days 00:30:00
Granularity report:


,granularity,from,to
0,0 days 00:30:00,2025-01-01,2060-06-30 23:30:00


Downsampling at least part of data to 60min
Missing dates after resampling count: 0
Chronological: True
Actual length: 271752 Expected length: 271752


,Date,Price
271742,2056-12-31 14:00:00,1.890
271743,2056-12-31 15:00:00,1.840
271744,2056-12-31 16:00:00,0.870
271745,2056-12-31 17:00:00,29.475
271746,2056-12-31 18:00:00,58.420
271747,2056-12-31 19:00:00,77.925
271748,2056-12-31 20:00:00,68.245
271749,2056-12-31 21:00:00,74.735
271750,2056-12-31 22:00:00,84.075
271751,2056-12-31 23:00:00,85.430
